In [ ]:
#!/usr/bin/env python3
"""
reproduce_results.py
--------------------
Reproduces evaluation metrics and qualitative predictions for the Multimodal Deepfake 
and Audio Spoof Detection pipeline (Model A: Visual, Model B: Audio, Model C: Cross-Fusion).
"""

import os
import sys
import argparse
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision import models
import timm


# ==========================================
# 1. REPRODUCIBILITY & SETUP
# ==========================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ==========================================
# 2. DATASET DEFINITION
# ==========================================
class FakeAVCelebDataset(Dataset):
    def __init__(self, csv_path, num_frames=5, transform=None):
        self.df = pd.read_csv(csv_path)
        if list(self.df.columns[:2]) != ["path", "label"]:
            self.df = self.df.iloc[:, :2]
            self.df.columns = ["path", "label"]
        self.items = self.df.values.tolist()
        self.num_frames = num_frames
        self.transform = transform or T.Compose([
            T.Resize((224, 224)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.items)

    def _load_frames(self, folder):
        folder = Path(folder)
        pts = sorted(folder.glob("frame_*.pt"))
        if len(pts) == 0:
            imgs = sorted(folder.glob("*.jpg"))
            if len(imgs) == 0:
                raise RuntimeError(f"No frame tensors or images found in {folder}")
            idxs = np.linspace(0, len(imgs) - 1, min(self.num_frames, len(imgs)), dtype=int)
            frames = []
            for i in idxs:
                im = Image.open(imgs[i]).convert("RGB")
                frames.append(self.transform(im))
            return torch.stack(frames)
        else:
            n = min(self.num_frames, len(pts))
            idxs = np.linspace(0, len(pts) - 1, n, dtype=int)
            frames = []
            for i in idxs:
                t = torch.load(pts[i], map_location="cpu")
                if not torch.is_tensor(t):
                    raise TypeError(f"Frame {pts[i]} is not a tensor.")
                frames.append(t)
            return torch.stack(frames)

    def _load_mel(self, mel_path):
        mel_loaded = torch.load(mel_path, map_location="cpu")
        if isinstance(mel_loaded, dict):
            if "mel_spec" in mel_loaded:
                mel = mel_loaded["mel_spec"].float()
            elif "mel_db" in mel_loaded:
                mel = mel_loaded["mel_db"].float()
            elif "mel" in mel_loaded:
                mel = mel_loaded["mel"].float()
            else:
                first = next(iter(mel_loaded.values()))
                mel = torch.tensor(first).float()
        elif torch.is_tensor(mel_loaded):
            mel = mel_loaded.float()
        else:
            raise TypeError(f"Unsupported mel format in {mel_path}")

        if mel.dim() == 2:
            mel = mel.unsqueeze(0)
        elif mel.dim() == 3 and mel.shape[0] != 1:
            mel = mel.mean(0, keepdim=True)

        mel = (mel - mel.mean()) / (mel.std(unbiased=False) + 1e-6)
        return mel

    def __getitem__(self, idx):
        folder, label = self.items[idx]
        frames = self._load_frames(folder)
        mel_path = Path(folder) / "mel.pt"
        if not mel_path.exists():
            raise RuntimeError(f"Missing mel.pt in {folder}")
        mel = self._load_mel(mel_path)
        return frames, mel, torch.tensor(int(label), dtype=torch.long)


def collate_variable(batch, max_mel_len=None):
    B = len(batch)
    frames = torch.stack([b[0] for b in batch], dim=0)  # [B, Nf, C, H, W]
    Ts = [b[1].shape[-1] for b in batch]
    Tmax = max(Ts) if max_mel_len is None else min(max(Ts), max_mel_len)
    mels = torch.zeros(B, 1, batch[0][1].shape[1], Tmax, dtype=torch.float32)
    for i, b in enumerate(batch):
        t = b[1].shape[-1]
        copy_t = min(t, Tmax)
        mels[i, :, :, :copy_t] = b[1][:, :, :copy_t]
    labels = torch.stack([b[2] for b in batch])
    return frames, mels, labels


# ==========================================
# 3. ARCHITECTURE DEFINITIONS
# ==========================================
class ModelA_Visual(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        self.cnn = models.efficientnet_b4(weights=None)
        self.cnn.classifier = nn.Identity()
        cnn_out = 1792

        self.vit = timm.create_model("vit_base_patch16_224", pretrained=False)
        self.vit.head = nn.Identity()
        trans_out = 768

        self.proj = nn.Linear(cnn_out + trans_out, embed_dim)
        self.visual_head = nn.Linear(embed_dim, 2)  # auxiliary classifier

    def forward(self, frames):
        # frames: [B, Nf, 3, 224, 224]
        B, Nf = frames.shape[0], frames.shape[1]
        x = frames.view(B * Nf, 3, 224, 224)
        cnn_feat = self.cnn(x)
        vit_feat = self.vit(x)
        fused = torch.cat([cnn_feat, vit_feat], dim=1)
        fused = fused.view(B, Nf, -1).mean(dim=1)
        emb = self.proj(fused)
        logits = self.visual_head(emb)
        return emb, logits


class ModelB_Audio(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1, 1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU()
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Linear(128, embed_dim)
        self.audio_head = nn.Linear(embed_dim, 2)  # auxiliary classifier

    def forward(self, mels):
        # mels: [B, 1, 80, T]
        x = self.cnn(mels)
        x = self.pool(x).flatten(1)
        emb = self.proj(x)
        logits = self.audio_head(emb)
        return emb, logits


class CrossAttentionFusion(nn.Module):
    def __init__(self, dim=256, heads=4, dropout=0.1):
        super().__init__()
        self.attn_va = nn.MultiheadAttention(embed_dim=dim, num_heads=heads, batch_first=True, dropout=dropout)
        self.attn_av = nn.MultiheadAttention(embed_dim=dim, num_heads=heads, batch_first=True, dropout=dropout)
        self.norm_v = nn.LayerNorm(dim)
        self.norm_a = nn.LayerNorm(dim)
        self.classifier = nn.Sequential(
            nn.Linear(dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2)
        )

    def forward(self, v_feat, a_feat):
        v_tok = v_feat.unsqueeze(1)
        a_tok = a_feat.unsqueeze(1)
        v2a, _ = self.attn_va(v_tok, a_tok, a_tok)
        a2v, _ = self.attn_av(a_tok, v_tok, v_tok)
        v_out = self.norm_v(v_tok + v2a).squeeze(1)
        a_out = self.norm_a(a_tok + a2v).squeeze(1)
        fused = (v_out + a_out) / 2.0
        logits = self.classifier(fused)
        return logits, v_out, a_out


class AVFusionModel(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        self.modelA = ModelA_Visual(embed_dim=embed_dim)
        self.modelB = ModelB_Audio(embed_dim=embed_dim)
        self.fusion = CrossAttentionFusion(dim=embed_dim, heads=4)

    def forward(self, frames, mels):
        v_emb, v_logits = self.modelA(frames)
        a_emb, a_logits = self.modelB(mels)
        fusion_logits, _, _ = self.fusion(v_emb, a_emb)
        return fusion_logits, v_logits, a_logits


# ==========================================
# 4. EVALUATION & TABLE UTILITIES
# ==========================================
def evaluate_pipeline(model, dataloader, device):
    model.eval()
    
    gt_labels = []
    fusion_preds, fusion_probs = [], []
    visual_preds, visual_probs = [], []
    audio_preds, audio_probs = [], []

    print("\n Running evaluation across all modalities...")
    with torch.no_grad():
        for frames, mels, labels in tqdm(dataloader, desc="Evaluating"):
            frames = frames.to(device).float()
            mels = mels.to(device).float()

            f_logits, v_logits, a_logits = model(frames, mels)

            # Probabilities & Predictions
            f_prob = F.softmax(f_logits, dim=1)[:, 1]
            v_prob = F.softmax(v_logits, dim=1)[:, 1]
            a_prob = F.softmax(a_logits, dim=1)[:, 1]

            gt_labels.extend(labels.cpu().numpy().tolist())
            
            fusion_probs.extend(f_prob.cpu().numpy().tolist())
            fusion_preds.extend(f_logits.argmax(dim=1).cpu().numpy().tolist())

            visual_probs.extend(v_prob.cpu().numpy().tolist())
            visual_preds.extend(v_logits.argmax(dim=1).cpu().numpy().tolist())

            audio_probs.extend(a_prob.cpu().numpy().tolist())
            audio_preds.extend(a_logits.argmax(dim=1).cpu().numpy().tolist())

    return (
        np.array(gt_labels),
        {"preds": np.array(fusion_preds), "probs": np.array(fusion_probs)},
        {"preds": np.array(visual_preds), "probs": np.array(visual_probs)},
        {"preds": np.array(audio_preds), "probs": np.array(audio_probs)},
    )


def compute_metrics(y_true, results, modality_name="Fusion"):
    y_pred = results["preds"]
    y_prob = results["probs"]

    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary")
    
    try:
        auc = roc_auc_score(y_true, y_prob)
    except Exception:
        auc = 0.0

    cm = confusion_matrix(y_true, y_pred)

    print(f"\n==================================================")
    print(f" PERFORMANCE METRICS: {modality_name.upper()}")
    print(f"==================================================")
    print(f"Accuracy  : {acc * 100:.2f}%")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1-Score  : {f1:.4f}")
    print(f"ROC AUC   : {auc:.4f}")
    print(f"\nConfusion Matrix:\n{cm}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=["Real", "Fake"], digits=4))

    return {"acc": acc, "prec": prec, "rec": rec, "f1": f1, "auc": auc}


def print_qualitative_table(y_true, v_res, a_res, f_res, n_samples=15):
    print(f"\n{'=' * 85}")
    print(f" QUALITATIVE PREDICTION TABLE (First {n_samples} Samples)")
    print(f"{'=' * 85}")
    print(f"{'GT':<8}{'Visual (Prob)':<25}{'Audio (Prob)':<25}{'Fusion Decision':<25}")
    print(f"{'-' * 85}")

    for i in range(min(n_samples, len(y_true))):
        gt = "Fake" if y_true[i] == 1 else "Real"
        
        v_label = "Fake" if v_res["preds"][i] == 1 else "Real"
        v_str = f"{v_label} ({v_res['probs'][i]:.2f})"
        
        a_label = "Fake" if a_res["preds"][i] == 1 else "Real"
        a_str = f"{a_label} ({a_res['probs'][i]:.2f})"
        
        f_label = "Fake" if f_res["preds"][i] == 1 else "Real"
        f_str = f"{f_label} ({f_res['probs'][i]:.2f})"

        print(f"{gt:<8}{v_str:<25}{a_str:<25}{f_str:<25}")

    print(f"{'=' * 85}\n")


# ==========================================
# 5. MAIN CLI EXECUTION
# ==========================================
def main():
    parser = argparse.ArgumentParser(description="Reproduce Model Evaluation Results")
    parser.add_argument("--metadata_csv", type=str, default="fakeavceleb_preprocessed_metadata.csv", help="Metadata CSV path")
    parser.add_argument("--checkpoint", type=str, default="modelC_option3c_crossfusion/modelC_crossfusion_best.pth", help="Checkpoint file path")
    parser.add_argument("--batch_size", type=int, default=8, help="Batch size for evaluation")
    parser.add_argument("--num_frames", type=int, default=5, help="Number of frames per video")
    parser.add_argument("--seed", type=int, default=42, help="Random seed")
    args = parser.parse_args()

    set_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f" Operating on device: {device}")

    # Load dataset
    if not Path(args.metadata_csv).exists():
        print(f" Error: Metadata CSV file '{args.metadata_csv}' not found.")
        sys.exit(1)

    print(f" Loading dataset metadata from {args.metadata_csv}...")
    dataset = FakeAVCelebDataset(csv_path=args.metadata_csv, num_frames=args.num_frames)
    dataloader = DataLoader(
        dataset,
        batch_size=args.batch_size,
        shuffle=False,
        collate_fn=collate_variable,
        num_workers=0
    )

    # Initialize model
    print(" Initializing AVFusionModel architecture...")
    model = AVFusionModel(embed_dim=256).to(device)

    # Load weights
    ckpt_path = Path(args.checkpoint)
    if ckpt_path.exists():
        print(f" Loading pre-trained weights from {ckpt_path}...")
        checkpoint = torch.load(ckpt_path, map_location=device)
        state_dict = checkpoint.get("model_state", checkpoint)
        model.load_state_dict(state_dict, strict=False)
        print(" Weights successfully loaded!")
    else:
        print(f" Warning: Checkpoint '{ckpt_path}' not found. Evaluating with initialized weights.")

    # Run evaluation
    y_true, fusion_res, visual_res, audio_res = evaluate_pipeline(model, dataloader, device)

    # Print quantitative metrics for all streams
    compute_metrics(y_true, visual_res, modality_name="Visual Stream (Model A)")
    compute_metrics(y_true, audio_res, modality_name="Audio Stream (Model B)")
    compute_metrics(y_true, fusion_res, modality_name="Cross-Attention Fusion (Model C)")

    # Print qualitative comparison table
    print_qualitative_table(y_true, visual_res, audio_res, fusion_res, n_samples=20)


if __name__ == "__main__":
    main()